# Alexa Dandridge - Homework 3

This assignment aims to improve the predictions for the Kaggle Homework (previously done hoemowork 2). I will explore three different sets of hyperparameter inputs for each of two different boosting models. 

In [22]:
pip install lightgbm catboost

Note: you may need to restart the kernel to use updated packages.


In [23]:
# Importing Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder #converts text to numbers
from sklearn.metrics import balanced_accuracy_score #main metric for evaluation
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [24]:
# importing training data
train = pd.read_csv('train.csv')
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [25]:
# importing testing data
test = pd.read_csv('test.csv') 
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [26]:
# saving test ids for later use
test_ids = test['id']

# dropping id column
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

# separating features and target
X = train.drop('Irrigation_Need', axis=1)

# encoding target labels into numbers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(train['Irrigation_Need'])

In [27]:
# identifying categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# converting them to category dtype
for col in categorical_cols:
    X[col] = X[col].astype('category')
    test[col] = test[col].astype('category')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

### Creating LightGBM 1

In [28]:
lgbm_1 = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbose=-1
)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

lgbm_cv_scores_1 = cross_val_score(
    lgbm_1,
    X,
    y,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

print("LightGBM Set 1 CV Scores:", lgbm_cv_scores_1)
print("LightGBM Set 1 Mean CV:", lgbm_cv_scores_1.mean())

lgbm_1.fit(X_train, y_train)
lgbm_pred_1 = lgbm_1.predict(X_val)

print("LightGBM Set 1 Validation Score:", balanced_accuracy_score(y_val, lgbm_pred_1))

LightGBM Set 1 CV Scores: [0.96121879 0.9617773  0.96160219]
LightGBM Set 1 Mean CV: 0.9615327585286977
LightGBM Set 1 Validation Score: 0.9645210682295241


### Creating LightGBM 2

In [29]:
lgbm_2 = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbose=-1
)

lgbm_cv_scores_2 = cross_val_score(
    lgbm_2,
    X,
    y,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

print("LightGBM Set 2 CV Scores:", lgbm_cv_scores_2)
print("LightGBM Set 2 Mean CV:", lgbm_cv_scores_2.mean())

lgbm_2.fit(X_train, y_train)
lgbm_pred_2 = lgbm_2.predict(X_val)

print("LightGBM Set 2 Validation Score:", balanced_accuracy_score(y_val, lgbm_pred_2))

LightGBM Set 2 CV Scores: [0.9615837  0.96229557 0.961844  ]
LightGBM Set 2 Mean CV: 0.9619077588097134
LightGBM Set 2 Validation Score: 0.9645733622966169


In [30]:
lgbm_3 = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=3,        # much shallower
    num_leaves=15,      # restrict complexity
    random_state=42,
    verbose=-1
)

lgbm_cv_scores_3 = cross_val_score(
    lgbm_3,
    X,
    y,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

print("LightGBM Set 3 CV Scores:", lgbm_cv_scores_3)
print("LightGBM Set 3 Mean CV:", lgbm_cv_scores_3.mean())

lgbm_3.fit(X_train, y_train)
lgbm_pred_3 = lgbm_3.predict(X_val)

print("LightGBM Set 3 Validation Score:", balanced_accuracy_score(y_val, lgbm_pred_3))

LightGBM Set 3 CV Scores: [0.95819726 0.95732601 0.95958009]
LightGBM Set 3 Mean CV: 0.9583677863611322
LightGBM Set 3 Validation Score: 0.9604494433995662


### Catboost 

In [31]:
# get indices of categorical columns
categorical_cols = X.select_dtypes(include=['category']).columns.tolist()
cat_features_idx = [X.columns.get_loc(col) for col in categorical_cols]

print("Categorical columns:", categorical_cols)

Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


### Catboost 1

In [32]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np

cat_1 = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

cv_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val_cv = y[train_idx], y[val_idx]

    cat_1.fit(X_tr, y_tr, cat_features=cat_features_idx)

    preds = cat_1.predict(X_val_cv)

    score = balanced_accuracy_score(y_val_cv, preds)
    cv_scores.append(score)

print("CatBoost Set 1 CV Scores:", cv_scores)
print("CatBoost Set 1 Mean CV:", np.mean(cv_scores))

CatBoost Set 1 CV Scores: [np.float64(0.9604721962933058), np.float64(0.9601666800721498), np.float64(0.9600778410227564)]
CatBoost Set 1 Mean CV: 0.9602389057960706


In [33]:
cat_1.fit(X_train, y_train, cat_features=cat_features_idx)

cat_pred_1 = cat_1.predict(X_val)

print("CatBoost Set 1 Validation Score:", balanced_accuracy_score(y_val, cat_pred_1))

CatBoost Set 1 Validation Score: 0.9635625987248922


### Catboost 2

In [34]:
cat_2 = CatBoostClassifier(
    iterations=10,
    depth=8,
    learning_rate=0.05,
    random_state=42,
    verbose=0
)

cv_scores_2 = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val_cv = y[train_idx], y[val_idx]

    cat_2.fit(X_tr, y_tr, cat_features=cat_features_idx)
    preds = cat_2.predict(X_val_cv)

    score = balanced_accuracy_score(y_val_cv, preds)
    cv_scores_2.append(score)

print("CatBoost Set 2 CV Scores:", cv_scores_2)
print("CatBoost Set 2 Mean CV:", np.mean(cv_scores_2))

cat_2.fit(X_train, y_train, cat_features=cat_features_idx)
cat_pred_2 = cat_2.predict(X_val)

print("CatBoost Set 2 Validation Score:", balanced_accuracy_score(y_val, cat_pred_2))

CatBoost Set 2 CV Scores: [np.float64(0.9562183257146898), np.float64(0.955019097840257), np.float64(0.9537293403116852)]
CatBoost Set 2 Mean CV: 0.9549889212888774
CatBoost Set 2 Validation Score: 0.9526750287601032


### Catboost 3

In [35]:
cat_3 = CatBoostClassifier(
    iterations=150,
    depth=4,              # shallower trees
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

cv_scores_3 = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val_cv = y[train_idx], y[val_idx]

    cat_3.fit(X_tr, y_tr, cat_features=cat_features_idx)
    preds = cat_3.predict(X_val_cv)

    score = balanced_accuracy_score(y_val_cv, preds)
    cv_scores_3.append(score)

print("CatBoost Set 3 CV Scores:", cv_scores_3)
print("CatBoost Set 3 Mean CV:", np.mean(cv_scores_3))

cat_3.fit(X_train, y_train, cat_features=cat_features_idx)
cat_pred_3 = cat_3.predict(X_val)

print("CatBoost Set 3 Validation Score:", balanced_accuracy_score(y_val, cat_pred_3))

CatBoost Set 3 CV Scores: [np.float64(0.9600243738664282), np.float64(0.9598440316694655), np.float64(0.9592277339933631)]
CatBoost Set 3 Mean CV: 0.9596987131764191
CatBoost Set 3 Validation Score: 0.9633342309855077


### Submitting lightgbm model to kaggle

In [36]:
# Best LightGBM model
best_lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbose=-1
)

# train on full training data
best_lgbm.fit(X, y)

# predict on test data
lgbm_test_preds = best_lgbm.predict(test)

# convert numeric predictions back to original labels
lgbm_test_labels = le.inverse_transform(lgbm_test_preds)

# create submission file
lgbm_submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': lgbm_test_labels
})

# save to csv
lgbm_submission.to_csv('lgbm_submission.csv', index=False)

print(lgbm_submission.head())

Exception ignored in: <function ResourceTracker.__del__ at 0x10412d800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes


       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


### Submitting a Catboost to Kaggle compeition

In [37]:
# Best CatBoost model
best_cat = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

# train on full training data
best_cat.fit(X, y, cat_features=cat_features_idx)

# predict on test data
cat_test_preds = best_cat.predict(test)

# CatBoost may return floats/shape issues, so flatten and convert to int
cat_test_preds = np.array(cat_test_preds).astype(int).ravel()

# convert numeric predictions back to original labels
cat_test_labels = le.inverse_transform(cat_test_preds)

# create submission file
cat_submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': cat_test_labels
})

# save to csv
cat_submission.to_csv('catboost_submission.csv', index=False)

print(cat_submission.head())

Exception ignored in: <function ResourceTracker.__del__ at 0x106fb5800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1066c5800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102a99800>
Traceback (most recent call last

       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


Exception ignored in: <function ResourceTracker.__del__ at 0x1068a1800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
